# Local partisan exposure

This notebook compares each persona's partisan exposure in the LLM-generated
networks with that same persona's ANES response. Outgoing nominations are the
primary network definition; reciprocal ties are a robustness check.

The comparison is intentionally **unweighted**: every generated network contains
the same 100 sampled personas, so the estimand is agreement for this fixed sample,
not a population estimate. LLM proportions are mapped to the five ANES response
categories with equal-width bins: `[0, .2]`, `(.2, .4]`, `(.4, .6]`,
`(.6, .8]`, and `(.8, 1]`. Egos with no ties in the relevant network view
are assigned zero Democratic, Republican, and other-party exposure. Egos with
ties but no usable alter-party information remain missing.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pickle
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from matplotlib import font_manager

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "code":
    BASE_DIR = BASE_DIR.parent

DATA_DIR = BASE_DIR / "outputs" / "data"
FIGURE_DIR = BASE_DIR / "outputs" / "figures"
TABLE_DIR = BASE_DIR / "outputs" / "tables"
NOTEBOOK_OUTPUT_DIR = BASE_DIR / "outputs" / "notebooks"
for folder in (DATA_DIR, FIGURE_DIR, TABLE_DIR, NOTEBOOK_OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

ANES_CSV = BASE_DIR / "data" / "anes_100.csv"
GRAPH_INPUTS = {
    "directed_out": (DATA_DIR / "graphs_sample_100_v3_directed.pkl", "out"),
    "mutual": (DATA_DIR / "graphs_sample_100_v3_mutual.pkl", "reciprocated"),
}
PLAUSIBILITY_CSV = DATA_DIR / "structural_plausibility_sample_100_v3_mutual.csv"

EXPOSURE_LONG_CSV = DATA_DIR / "local_evaluation_exposure_long_sample_100_v3_by_network_view.csv"
VIEW_METRICS_CSV = DATA_DIR / "local_evaluation_metrics_by_network_view_sample_100_v3.csv"
SUMMARY_CSV = DATA_DIR / "local_evaluation_summary_sample_100_v3_by_network_view.csv"
VIEW_COVERAGE_CSV = DATA_DIR / "local_evaluation_coverage_by_network_view_sample_100_v3.csv"
PAIRED_VIEW_CSV = DATA_DIR / "local_evaluation_paired_view_differences_sample_100_v3.csv"
PLAUSIBILITY_EXPOSURE_CSV = DATA_DIR / "local_evaluation_by_structural_plausibility_sample_100_v3.csv"

PRIMARY_NETWORK_VIEW = "directed_out"
VOTE_ATTR = "w2presvtwho"
SURVEY_DEM_COL = "nd_fd"
SURVEY_REP_COL = "nd_fr"
BIN_CATS = [1, 2, 3, 4, 5]
BIN_EDGES = [0, .2, .4, .6, .8, 1.0]
METHOD_ORDER = ["c1", "c5", "c10", "c25", "c50"]
EGO_PARTY_ORDER = ["dem", "rep", "other"]

MODEL_LABELS = {
    "llama-31-8b-instruct": "Llama-3.1 8B",
    "meta_llama_31_70b_instruct_awq_int4": "Llama-3.1 70B AWQ",
    "mistral-7b-instruct-v03": "Mistral 7B",
    "qwen25-7b-instruct": "Qwen2.5 7B",
    "qwen25_72b_instruct_awq": "Qwen2.5 72B AWQ",
}
MODEL_ORDER = [
    "Llama-3.1 8B", "Mistral 7B", "Qwen2.5 7B",
    "Llama-3.1 70B AWQ", "Qwen2.5 72B AWQ",
]
ORDER_LABELS = {
    "gender_race_education_location_age_religion_vote": "vote-last",
    "vote_religion_age_location_education_race_gender": "vote-first",
}

sns.set_theme(style="whitegrid", context="notebook")

font_dir = BASE_DIR / "font"
barlow_files = sorted(font_dir.glob("Barlow-*.ttf")) if font_dir.exists() else []
for font_path in barlow_files:
    font_manager.fontManager.addfont(str(font_path))
if not barlow_files:
    raise FileNotFoundError(f"Barlow font files not found in {font_dir}")

plt.rcParams.update({
    "font.family": "Barlow",
    "font.weight": "normal",
    "axes.labelweight": "normal",
    "axes.titleweight": "normal",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 110,
})


## Definitions and validation helpers

`other` parties remain in the denominator, while missing alter party is excluded.
This makes Democratic and Republican exposure shares interpretable as shares of
all alters with usable party information.


In [ ]:
DEMOCRAT_VALUES = {2, "2", "Democrat", "are Democrat", "Joe Biden", "Biden"}
REPUBLICAN_VALUES = {1, "1", "Republican", "are Republican", "Donald Trump", "Trump"}


def normalize_party(value):
    if pd.isna(value):
        return np.nan
    if value in DEMOCRAT_VALUES:
        return "dem"
    if value in REPUBLICAN_VALUES:
        return "rep"
    text = str(value).strip().lower()
    if text in {"democrat", "are democrat", "joe biden", "biden"}:
        return "dem"
    if text in {"republican", "are republican", "donald trump", "trump"}:
        return "rep"
    return "other"


def parse_run_id(run_id):
    stem = Path(str(run_id)).stem
    match = re.match(r"(.+)_temp([0-9.]+)(?:_chunk(\d+))?_run(\d+)_(.+)$", stem)
    if match is None:
        raise ValueError(f"Cannot parse run identifier: {run_id}")
    parsed_model, temperature, chunk, run_number, attribute_order = match.groups()
    chunk_size = int(chunk) if chunk is not None else -1
    return {
        "parsed_model": parsed_model,
        "temperature": float(temperature),
        "chunk_size": chunk_size,
        "method": "c1" if chunk_size < 0 else f"c{chunk_size}",
        "run_number": int(run_number),
        "attribute_order": attribute_order,
        "attr_label": ORDER_LABELS.get(attribute_order, "mixed"),
    }


def clean_survey_category(series):
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.where(numeric.isin(BIN_CATS)).astype("Int64")


def bin_exposure(series):
    # Map shares to five ANES categories using equal-width, right-closed bins.
    return pd.cut(
        pd.to_numeric(series, errors="coerce"),
        bins=BIN_EDGES,
        labels=BIN_CATS,
        include_lowest=True,
        right=True,
    ).astype("Int64")


def neighbors_for_exposure(graph, node, mode):
    if mode == "out":
        return list(graph.successors(node)) if graph.is_directed() else list(graph.neighbors(node))
    if mode == "reciprocated":
        if graph.is_directed():
            return list(set(graph.successors(node)).intersection(graph.predecessors(node)))
        return list(graph.neighbors(node))
    raise ValueError(f"Unknown exposure mode: {mode}")


def compute_node_exposure(graph, mode):
    rows = []
    for ego in graph.nodes:
        neighbors = neighbors_for_exposure(graph, ego, mode)
        counts = {"dem": 0, "rep": 0, "other": 0}
        for alter in neighbors:
            party = normalize_party(graph.nodes[alter].get(VOTE_ATTR))
            if not pd.isna(party):
                counts[party] += 1
        valid = sum(counts.values())
        if valid:
            shares = {party: count / valid for party, count in counts.items()}
        elif not neighbors:
            # No outgoing (or mutual) ties means zero exposure to every party.
            shares = {party: 0.0 for party in counts}
        else:
            # Ties exist, but none has usable party information.
            shares = {party: np.nan for party in counts}
        rows.append({
            "caseid": graph.nodes[ego].get("caseid", ego),
            "ego_party": normalize_party(graph.nodes[ego].get(VOTE_ATTR)),
            "n_neighbors": len(neighbors),
            "n_valid_party_neighbors": valid,
            "llm_pct_dem": shares["dem"],
            "llm_pct_rep": shares["rep"],
            "llm_pct_other": shares["other"],
        })
    result = pd.DataFrame(rows)
    result["llm_cat_dem"] = bin_exposure(result["llm_pct_dem"])
    result["llm_cat_rep"] = bin_exposure(result["llm_pct_rep"])
    return result


def probability_vector(values):
    values = pd.to_numeric(pd.Series(values), errors="coerce").dropna().astype(int)
    counts = values.value_counts().reindex(BIN_CATS, fill_value=0).sort_index()
    return counts / counts.sum() if counts.sum() else pd.Series(np.nan, index=BIN_CATS)


def cumulative_distribution(values):
    return probability_vector(values).cumsum()


In [ ]:
graphs_by_view = {}
for network_view, (graph_path, exposure_mode) in GRAPH_INPUTS.items():
    with open(graph_path, "rb") as handle:
        graphs_by_view[network_view] = pickle.load(handle)

survey = pd.read_csv(ANES_CSV, encoding="mac_roman", low_memory=False)[
    ["caseid", SURVEY_DEM_COL, SURVEY_REP_COL, VOTE_ATTR]
].copy()
survey["caseid"] = pd.to_numeric(survey["caseid"], errors="raise").astype(int)
survey["survey_cat_dem"] = clean_survey_category(survey[SURVEY_DEM_COL])
survey["survey_cat_rep"] = clean_survey_category(survey[SURVEY_REP_COL])
survey["ego_party"] = survey[VOTE_ATTR].apply(normalize_party)

assert len(survey) == survey["caseid"].nunique() == 100
assert set(graphs_by_view) == {"directed_out", "mutual"}
assert set(graphs_by_view["directed_out"]) == set(graphs_by_view["mutual"])

survey_ids = set(survey["caseid"])
survey_party = survey.set_index("caseid")["ego_party"].to_dict()
validated_runs = 0
for model in graphs_by_view["directed_out"]:
    directed_runs = graphs_by_view["directed_out"][model]
    mutual_runs = graphs_by_view["mutual"][model]
    assert set(directed_runs) == set(mutual_runs)
    for run_id, directed in directed_runs.items():
        mutual = mutual_runs[run_id]
        assert directed.is_directed() and not mutual.is_directed()
        assert directed.number_of_nodes() == mutual.number_of_nodes() == 100
        directed_ids = {int(directed.nodes[n].get("caseid", n)) for n in directed.nodes}
        mutual_ids = {int(mutual.nodes[n].get("caseid", n)) for n in mutual.nodes}
        assert directed_ids == mutual_ids == survey_ids
        for node in directed.nodes:
            caseid = int(directed.nodes[node].get("caseid", node))
            assert normalize_party(directed.nodes[node].get(VOTE_ATTR)) == survey_party[caseid]
        reciprocal = {
            frozenset((u, v)) for u, v in directed.edges()
            if u != v and directed.has_edge(v, u)
        }
        mutual_edges = {frozenset((u, v)) for u, v in mutual.edges() if u != v}
        assert reciprocal == mutual_edges
        validated_runs += 1

run_counts = {
    view: sum(len(runs) for runs in models.values())
    for view, models in graphs_by_view.items()
}
assert run_counts == {"directed_out": 1000, "mutual": 1000}
print(f"Validated {validated_runs:,} paired runs, 100 personas per run.")
print("Survey valid responses:", {
    "Democratic exposure": int(survey["survey_cat_dem"].notna().sum()),
    "Republican exposure": int(survey["survey_cat_rep"].notna().sum()),
})


## Persona-level exposure data

Notebook 02 classifies each mutual network using all five structural plausibility
criteria. Those flags are joined here by the complete experimental key; the same
flag is attached to the matching directed and mutual views of a run.


In [ ]:
plausibility = pd.read_csv(PLAUSIBILITY_CSV)
plausibility = plausibility.rename(columns={
    "configuration": "method", "temp": "temperature", "run": "run_number"
})
plausibility["attr_label"] = plausibility["order"].map({
    "G-first": "vote-last", "V-first": "vote-first"
})
plausibility_key = ["model", "temperature", "method", "attr_label", "run_number"]
assert len(plausibility) == plausibility[plausibility_key].drop_duplicates().shape[0] == 1000
plausibility = plausibility[plausibility_key + ["plausible_all_5"]]

records = []
for network_view, models in graphs_by_view.items():
    exposure_mode = GRAPH_INPUTS[network_view][1]
    for model, run_graphs in models.items():
        for run_id, graph in run_graphs.items():
            metadata = parse_run_id(run_id)
            node_exposure = compute_node_exposure(graph, exposure_mode)
            node_exposure["network_view"] = network_view
            node_exposure["model"] = model
            node_exposure["model_label"] = MODEL_LABELS.get(model, model)
            node_exposure["run_id"] = str(run_id)
            for name, value in metadata.items():
                node_exposure[name] = value
            records.append(node_exposure)

llm_nodes = pd.concat(records, ignore_index=True)
llm_nodes["caseid"] = pd.to_numeric(llm_nodes["caseid"], errors="raise").astype(int)
llm_nodes = llm_nodes.merge(plausibility, on=plausibility_key, how="left", validate="many_to_one")
assert llm_nodes["plausible_all_5"].notna().all()
llm_nodes["structurally_plausible"] = llm_nodes.pop("plausible_all_5").astype(bool)

exposure_columns = ["llm_pct_dem", "llm_pct_rep", "llm_pct_other"]
zero_tie_rows = llm_nodes["n_neighbors"].eq(0)
no_party_data_rows = llm_nodes["n_neighbors"].gt(0) & llm_nodes["n_valid_party_neighbors"].eq(0)
assert llm_nodes.loc[zero_tie_rows, exposure_columns].eq(0).all().all()
assert llm_nodes.loc[no_party_data_rows, exposure_columns].isna().all().all()

node_df = llm_nodes.merge(
    survey[["caseid", "survey_cat_dem", "survey_cat_rep"]],
    on="caseid", how="left", validate="many_to_one",
)
base_columns = [
    "network_view", "model", "model_label", "temperature", "chunk_size",
    "method", "attr_label", "attribute_order", "run_number", "run_id",
    "caseid", "ego_party", "n_neighbors", "n_valid_party_neighbors",
    "llm_pct_other",
    "structurally_plausible",
]

parts = []
for party in ("dem", "rep"):
    part = node_df[
        base_columns + [f"llm_pct_{party}", f"llm_cat_{party}", f"survey_cat_{party}"]
    ].copy()
    part = part.rename(columns={
        f"llm_pct_{party}": "llm_pct",
        f"llm_cat_{party}": "llm_cat",
        f"survey_cat_{party}": "survey_cat",
    })
    part["party"] = party
    parts.append(part)

exposure_long_all = pd.concat(parts, ignore_index=True)
for column in ("llm_cat", "survey_cat"):
    exposure_long_all[column] = pd.to_numeric(exposure_long_all[column], errors="coerce")
exposure_long_all["signed_error"] = exposure_long_all["llm_cat"] - exposure_long_all["survey_cat"]
exposure_long_all["abs_error"] = exposure_long_all["signed_error"].abs()
exposure_long_all["setting"] = (
    exposure_long_all["model_label"]
    + " | T=" + exposure_long_all["temperature"].astype(str)
    + " | " + exposure_long_all["method"]
    + " | " + exposure_long_all["attr_label"]
)

output_columns = [
    "network_view", "model", "model_label", "temperature", "chunk_size",
    "method", "attr_label", "attribute_order", "run_number", "run_id",
    "caseid", "ego_party", "n_neighbors", "n_valid_party_neighbors",
    "llm_pct", "llm_pct_other", "llm_cat", "survey_cat", "party", "setting",
    "signed_error", "abs_error", "structurally_plausible",
]
exposure_long_all[output_columns].to_csv(EXPOSURE_LONG_CSV, index=False)
exposure_long = exposure_long_all.query("network_view == @PRIMARY_NETWORK_VIEW").copy()

print(f"Saved {len(exposure_long_all):,} persona-target observations to {EXPOSURE_LONG_CSV.name}.")
print("Structurally plausible unique networks:",
      exposure_long.drop_duplicates(["model", "run_id"])["structurally_plausible"].value_counts().to_dict())


## Run-level evaluation

Every metric is computed within one generated network first. Means and standard
errors are then calculated across the ten independent runs, so dense networks or
configurations do not receive extra weight. Wasserstein and KS are ordinal
distribution distances on the five-category scale.


In [ ]:
def summarize_run(group):
    paired = group[["llm_cat", "survey_cat"]].dropna()
    if paired.empty:
        return pd.Series({name: np.nan for name in [
            "n", "mean_llm_cat", "mean_survey_cat", "signed_error",
            "mae", "rmse", "match_rate", "wasserstein", "ks",
        ]})
    error = paired["llm_cat"] - paired["survey_cat"]
    llm_prob = probability_vector(paired["llm_cat"])
    survey_prob = probability_vector(paired["survey_cat"])
    cdf_difference = llm_prob.cumsum() - survey_prob.cumsum()
    return pd.Series({
        "n": len(paired),
        "mean_llm_cat": paired["llm_cat"].mean(),
        "mean_survey_cat": paired["survey_cat"].mean(),
        "signed_error": error.mean(),
        "mae": error.abs().mean(),
        "rmse": np.sqrt(np.square(error).mean()),
        "match_rate": error.eq(0).mean(),
        "wasserstein": cdf_difference.abs().sum(),
        "ks": cdf_difference.abs().max(),
    })


run_group_columns = [
    "network_view", "model", "model_label", "temperature", "chunk_size",
    "method", "attr_label", "attribute_order", "run_number", "run_id",
    "structurally_plausible", "party", "ego_party",
]
run_metrics = (
    exposure_long_all
    .groupby(run_group_columns, dropna=False, observed=True)
    .apply(summarize_run, include_groups=False)
    .reset_index()
)
assert len(run_metrics) == 2 * 1000 * 2 * 3
run_metrics.to_csv(VIEW_METRICS_CSV, index=False)

summary_groups = [
    "network_view", "model", "model_label", "temperature", "method",
    "attr_label", "party", "ego_party",
]
metric_summary = (
    run_metrics.groupby(summary_groups, observed=True)
    .agg(
        n_runs=("signed_error", "count"),
        signed_error_mean=("signed_error", "mean"),
        signed_error_sd=("signed_error", "std"),
        mae_mean=("mae", "mean"),
        mae_sd=("mae", "std"),
        rmse_mean=("rmse", "mean"),
        wasserstein_mean=("wasserstein", "mean"),
        ks_mean=("ks", "mean"),
    )
    .reset_index()
)
metric_summary["signed_error_se"] = metric_summary["signed_error_sd"] / np.sqrt(metric_summary["n_runs"])
metric_summary["mae_se"] = metric_summary["mae_sd"] / np.sqrt(metric_summary["n_runs"])
assert metric_summary["n_runs"].eq(10).all()
metric_summary.to_csv(SUMMARY_CSV, index=False)

coverage_groups = [
    "network_view", "model", "model_label", "temperature", "chunk_size",
    "method", "attr_label", "attribute_order", "run_number", "run_id",
    "structurally_plausible",
]
network_view_coverage = (
    llm_nodes.groupby(coverage_groups, observed=True)
    .agg(
        n_personas=("caseid", "nunique"),
        mean_neighbors=("n_neighbors", "mean"),
        zero_neighbor_rate=("n_neighbors", lambda x: x.eq(0).mean()),
        valid_party_alter_rate=("n_valid_party_neighbors", lambda x: x.gt(0).mean()),
        usable_exposure_rate=("llm_pct_dem", lambda x: x.notna().mean()),
    )
    .reset_index()
)
network_view_coverage.to_csv(VIEW_COVERAGE_CSV, index=False)

display(
    run_metrics.groupby(["network_view", "party", "ego_party"])[
        ["signed_error", "mae", "wasserstein", "ks"]
    ].mean().round(3)
)
display(
    network_view_coverage.groupby("network_view")[[
        "mean_neighbors", "zero_neighbor_rate", "valid_party_alter_rate",
        "usable_exposure_rate"
    ]].mean().round(3)
)


## Tables S7 and S8: complete nomination and reciprocal mean signed deviations


In [ ]:
def make_complete_msd_table(network_view, representation, output_name, label):
    table_summary = (
        run_metrics.query("network_view == @network_view")
        .groupby(["model_label", "temperature", "method", "attr_label", "ego_party", "party"], observed=True)
        .agg(mean=("signed_error", "mean"), sd=("signed_error", "std"), n=("signed_error", "count"))
        .reset_index()
    )
    table_summary["se"] = table_summary["sd"] / np.sqrt(table_summary["n"])
    table_summary["value"] = table_summary.apply(
        lambda row: "--" if pd.isna(row["mean"]) else f'{row["mean"]:.3f} ({row["se"]:.3f})',
        axis=1,
    )
    table = (
        table_summary.pivot(
            index=["model_label", "temperature", "method", "attr_label", "ego_party"],
            columns="party", values="value",
        )
        .reset_index()
        .rename(columns={
            "model_label": "Model", "temperature": "T", "method": "Method",
            "attr_label": "Order", "ego_party": "Ego",
            "dem": "Dem. exposure", "rep": "Rep. exposure",
        })
    )
    table["Model"] = pd.Categorical(table["Model"], MODEL_ORDER, ordered=True)
    table["Method"] = pd.Categorical(table["Method"], METHOD_ORDER, ordered=True)
    table["T"] = table["T"].map(lambda value: f"{value:g}")
    table["Order"] = table["Order"].map({"vote-first": "First", "vote-last": "Last"})
    table["Ego"] = pd.Categorical(
        table["Ego"].map({"dem": "Democratic", "rep": "Republican", "other": "Other"}),
        ["Democratic", "Republican", "Other"], ordered=True,
    )
    table = table.sort_values(["Model", "T", "Method", "Order", "Ego"])
    caption = (
        rf"\textbf{{Mean Signed Deviation between LLM-generated and survey-reported partisan exposure "
        rf"in {representation} networks.}} Values report the unweighted mean across 10 runs for each "
        r"experimental configuration, with standard errors in parentheses. Positive values indicate "
        r"overestimation relative to the corresponding ANES personas, while negative values indicate "
        r"underestimation. Order indicates whether vote was presented first or last in the persona description."
    )
    latex = table.to_latex(
        index=False, escape=False, longtable=True, column_format="@{}lllllcc@{}",
        caption=caption, label=label,
    )
    output_path = BASE_DIR / "code" / "tables" / output_name
    output_path.write_text(latex)
    print(f"Saved {output_path.relative_to(BASE_DIR)} with {len(table):,} rows.")
    return table


table_s7 = make_complete_msd_table(
    "directed_out", "nomination", "table_S7_partisan_exposure_nomination.tex",
    "tab:complete_msd_nomination",
)
table_s8 = make_complete_msd_table(
    "mutual", "reciprocal", "table_S8_partisan_exposure_reciprocal.tex",
    "tab:complete_msd_reciprocal",
)
display(table_s7.head(12))
display(table_s8.head(12))


## Does exposure bias remain among structurally plausible networks?

This is a robustness comparison, not a configuration-selection step. The
`plausible` subset contains networks meeting all five notebook-02 structural
criteria; `all` retains the full experimental design.


In [ ]:
primary_metrics = run_metrics.query("network_view == @PRIMARY_NETWORK_VIEW").copy()
scope_frames = []
for scope, frame in {
    "all": primary_metrics,
    "plausible": primary_metrics.query("structurally_plausible"),
}.items():
    overall = (
        frame.groupby(["party", "ego_party"], observed=True)
        .agg(
            n_runs=("signed_error", "count"),
            signed_error_mean=("signed_error", "mean"),
            signed_error_sd=("signed_error", "std"),
            mae_mean=("mae", "mean"),
            mae_sd=("mae", "std"),
        ).reset_index()
    )
    overall["model_label"] = "All models"
    by_model = (
        frame.groupby(["model_label", "party", "ego_party"], observed=True)
        .agg(
            n_runs=("signed_error", "count"),
            signed_error_mean=("signed_error", "mean"),
            signed_error_sd=("signed_error", "std"),
            mae_mean=("mae", "mean"),
            mae_sd=("mae", "std"),
        ).reset_index()
    )
    combined = pd.concat([overall, by_model], ignore_index=True)
    combined["scope"] = scope
    scope_frames.append(combined)

plausibility_exposure = pd.concat(scope_frames, ignore_index=True)
plausibility_exposure["signed_error_se"] = (
    plausibility_exposure["signed_error_sd"] / np.sqrt(plausibility_exposure["n_runs"])
)
plausibility_exposure["mae_se"] = (
    plausibility_exposure["mae_sd"] / np.sqrt(plausibility_exposure["n_runs"])
)
plausibility_exposure.to_csv(PLAUSIBILITY_EXPOSURE_CSV, index=False)
display(
    plausibility_exposure.query("model_label == 'All models'")
    [["scope", "party", "ego_party", "n_runs", "signed_error_mean", "mae_mean"]]
    .round(3)
)


## Paired directed-out versus mutual robustness check

The two views are differenced within the same model, configuration, ordering,
temperature, run, exposure target, and ego group. A positive delta means that
restricting exposure to reciprocal ties increases the metric.


In [ ]:
pair_keys = [
    "model", "model_label", "temperature", "chunk_size", "method", "attr_label",
    "attribute_order", "run_number", "run_id", "structurally_plausible",
    "party", "ego_party",
]
paired_views = run_metrics.pivot(
    index=pair_keys, columns="network_view", values=["signed_error", "mae"]
)
paired_views.columns = [f"{metric}_{view}" for metric, view in paired_views.columns]
paired_views = paired_views.reset_index()
paired_views["mutual_minus_directed_signed_error"] = (
    paired_views["signed_error_mutual"] - paired_views["signed_error_directed_out"]
)
paired_views["mutual_minus_directed_mae"] = (
    paired_views["mae_mutual"] - paired_views["mae_directed_out"]
)
assert len(paired_views) == 1000 * 2 * 3
paired_views.to_csv(PAIRED_VIEW_CSV, index=False)

paired_summary = (
    paired_views.groupby(["party", "ego_party"], observed=True)
    .agg(
        n_pairs=("mutual_minus_directed_signed_error", "count"),
        signed_delta=("mutual_minus_directed_signed_error", "mean"),
        signed_delta_sd=("mutual_minus_directed_signed_error", "std"),
        mae_delta=("mutual_minus_directed_mae", "mean"),
        mae_delta_sd=("mutual_minus_directed_mae", "std"),
    ).reset_index()
)
paired_summary["signed_delta_se"] = paired_summary["signed_delta_sd"] / np.sqrt(paired_summary["n_pairs"])
paired_summary["mae_delta_se"] = paired_summary["mae_delta_sd"] / np.sqrt(paired_summary["n_pairs"])
display(paired_summary.round(3))

plot_pairs = paired_summary.query("ego_party in ['dem', 'rep']").copy()
plot_pairs["comparison"] = (
    plot_pairs["ego_party"].map({"dem": "Dem ego", "rep": "Rep ego"})
    + " → " + plot_pairs["party"].map({"dem": "Dem exposure", "rep": "Rep exposure"})
)
comparison_order = [
    "Dem ego → Dem exposure", "Dem ego → Rep exposure",
    "Rep ego → Dem exposure", "Rep ego → Rep exposure",
]
plot_pairs["comparison"] = pd.Categorical(plot_pairs["comparison"], comparison_order, ordered=True)
plot_pairs = plot_pairs.sort_values("comparison")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), sharex=True)
for ax, mean_col, se_col, title in [
    (axes[0], "signed_delta", "signed_delta_se", "Change in signed error"),
    (axes[1], "mae_delta", "mae_delta_se", "Change in absolute error"),
]:
    positions = np.arange(len(plot_pairs))
    ax.errorbar(
        positions, plot_pairs[mean_col], yerr=1.96 * plot_pairs[se_col],
        fmt="o", color="#5b4b8a", capsize=4, linewidth=1.5,
    )
    ax.axhline(0, color="0.35", linestyle="--", linewidth=1)
    ax.set_xticks(positions, plot_pairs["comparison"], rotation=25, ha="right")
    ax.set_title(title)
    ax.set_ylabel("Mutual − directed-out")
fig.suptitle("Paired effect of restricting exposure to reciprocal ties", y=1.02)
fig.tight_layout()
for suffix, kwargs in [("pdf", {}), ("png", {"dpi": 300})]:
    fig.savefig(FIGURE_DIR / f"Fig_local_exposure_directed_out_vs_mutual.{suffix}",
                bbox_inches="tight", **kwargs)
plt.show()


## Main exposure figures

The earlier publication layout is retained: annotated configuration heatmaps
above a model-by-model CDF grid. The complete figure is produced separately for
the primary directed-out networks and for the mutual-network robustness view.
Both use the same heatmap scale and the same model colors, permitting direct
visual comparison. Thin CDF curves are individual network runs; thick curves
are equally weighted run averages. The black curve is the fixed ANES reference.


In [ ]:
EXPOSURE_COLORS = {"dem": "#2166ac", "rep": "#b2182b"}
SURVEY_COLOR = "#111111"
FIG_FONT_WEIGHT = "normal"
EGO_LABELS = {"dem": "Dem", "rep": "Rep", "other": "Other"}
MODEL_SIZE_GROUPS = [
    ("Smaller models (7–8B)", ["Llama-3.1 8B", "Mistral 7B", "Qwen2.5 7B"]),
    ("Larger models (70–72B)", ["Llama-3.1 70B AWQ", "Qwen2.5 72B AWQ"]),
]
VIEW_LABELS = {"directed_out": "Nomination networks", "mutual": "Reciprocal networks"}


def contiguous_spans(labels):
    spans = []
    start = 0
    for index in range(1, len(labels) + 1):
        if index == len(labels) or labels[index] != labels[start]:
            spans.append((labels[start], start, index))
            start = index
    return spans


def make_exposure_figure(network_view):
    view_metrics = run_metrics[
        run_metrics["network_view"].eq(network_view)
        & run_metrics["model_label"].isin(MODEL_ORDER)
    ].copy()
    view_exposure = exposure_long_all[
        exposure_long_all["network_view"].eq(network_view)
        & exposure_long_all["model_label"].isin(MODEL_ORDER)
    ].copy()

    model_order = [model for model in MODEL_ORDER if model in set(view_metrics["model_label"])]
    model_group_ranges = []
    for group_label, group_models in MODEL_SIZE_GROUPS:
        indices = [model_order.index(model) for model in group_models if model in model_order]
        if indices:
            model_group_ranges.append((group_label, min(indices), max(indices)))
    size_model_boundaries = [end + 1 for _, _, end in model_group_ranges[:-1]]

    view_metrics["method"] = pd.Categorical(
        view_metrics["method"].astype(str), METHOD_ORDER, ordered=True
    )
    view_metrics["row_label"] = (
        view_metrics["model_label"] + " | " + view_metrics["method"].astype(str)
    )
    view_metrics["column_label"] = (
        "T=" + view_metrics["temperature"].map(lambda x: f"{x:g}")
        + " | " + view_metrics["attr_label"].astype(str)
        + " | " + view_metrics["ego_party"].map(EGO_LABELS).astype(str)
    )

    row_order_frame = view_metrics[["model_label", "method", "row_label"]].drop_duplicates()
    row_order_frame["model_label"] = pd.Categorical(
        row_order_frame["model_label"], model_order, ordered=True
    )
    row_order = row_order_frame.sort_values(["model_label", "method"])["row_label"].tolist()
    column_order = (
        view_metrics[["temperature", "attr_label", "ego_party", "column_label"]]
        .drop_duplicates()
        .assign(
            attr_rank=lambda frame: frame["attr_label"].map({"vote-first": 0, "vote-last": 1}),
            ego_rank=lambda frame: frame["ego_party"].map({"dem": 0, "other": 1, "rep": 2}),
        )
        .sort_values(["temperature", "attr_rank", "ego_rank"])["column_label"]
        .tolist()
    )

    combined_row_order = []
    spacer_rows = []
    for group_index, (_, start, end) in enumerate(model_group_ranges):
        if group_index > 0:
            spacer = f"__size_spacer_{group_index}__"
            spacer_rows.append(spacer)
            combined_row_order.append(spacer)
        group_models = set(model_order[start:end + 1])
        combined_row_order.extend(
            row for row in row_order if row.split(" | ")[0] in group_models
        )

    display_row_labels = [
        "" if row in spacer_rows else row.split(" | ")[1]
        for row in combined_row_order
    ]
    model_row_spans = []
    for model_label in model_order:
        positions = [
            index for index, row in enumerate(combined_row_order)
            if row not in spacer_rows and row.split(" | ")[0] == model_label
        ]
        model_row_spans.append((model_label, min(positions), max(positions) + 1))

    column_parts = [label.split(" | ") for label in column_order]
    ego_tick_labels = [parts[2] for parts in column_parts]
    attribute_spans = contiguous_spans([parts[1] for parts in column_parts])
    temperature_spans = contiguous_spans([parts[0] for parts in column_parts])
    attribute_boundaries = [end for _, _, end in attribute_spans[:-1]]
    temperature_boundaries = [end for _, _, end in temperature_spans[:-1]]

    fig = plt.figure(figsize=(22.5, 25.2), constrained_layout=False)
    outer = fig.add_gridspec(2, 1, height_ratios=[1.42, .94], hspace=.46)

    # Panel a: annotated heatmaps with the original grouped labels.
    heat_grid = outer[0].subgridspec(1, 3, width_ratios=[1, 1, .018], wspace=.045)
    heat_axes = []
    colorbar_axis = fig.add_subplot(heat_grid[0, 2])
    for panel_index, (party, title) in enumerate([
        ("dem", "Democratic exposure"), ("rep", "Republican exposure")
    ]):
        matrix = (
            view_metrics[view_metrics["party"].eq(party)]
            .pivot_table(
                index="row_label", columns="column_label",
                values="signed_error", aggfunc="mean", observed=True,
            )
            .reindex(index=combined_row_order, columns=column_order)
        )
        axis = fig.add_subplot(
            heat_grid[0, panel_index], sharey=heat_axes[0] if heat_axes else None
        )
        heat_axes.append(axis)
        sns.heatmap(
            matrix, mask=matrix.isna(), annot=True, fmt=".2f",
            cmap="PiYG", center=0, vmin=-1.8, vmax=1.8,
            linewidths=.34, linecolor="white",
            cbar=(panel_index == 1),
            cbar_ax=colorbar_axis if panel_index == 1 else None,
            annot_kws={"size": 16, "weight": FIG_FONT_WEIGHT, "fontfamily": "Barlow"},
            ax=axis,
        )
        axis.set_facecolor("white")
        axis.set_title(title, fontsize=30, pad=20, fontweight=FIG_FONT_WEIGHT)
        axis.set_xlabel("")
        axis.set_ylabel("")
        axis.set_xticks(np.arange(len(column_order)) + .5)
        axis.set_xticklabels(ego_tick_labels, rotation=0)
        axis.tick_params(axis="x", labelsize=20, pad=5, colors="#333333")
        axis.set_yticks(np.arange(len(display_row_labels)) + .5)
        if panel_index == 0:
            axis.set_yticklabels(display_row_labels, rotation=0)
            axis.tick_params(axis="y", labelsize=24, pad=6, colors="#333333")
        else:
            axis.tick_params(axis="y", labelleft=False, length=0)

        if panel_index == 0:
            for model_label, start, end in model_row_spans:
                center = (start + end) / 2
                axis.text(
                    -.150, center, model_label,
                    transform=axis.get_yaxis_transform(), ha="right", va="center",
                    fontsize=24, fontweight="semibold", color="#333333", clip_on=False,
                )
                axis.plot(
                    [-.088, -.102, -.102, -.088],
                    [start + .15, start + .15, end - .15, end - .15],
                    transform=axis.get_yaxis_transform(), color="#555555",
                    linewidth=1.2, clip_on=False,
                )

        for attribute_label, start, end in attribute_spans:
            center = (start + end) / 2
            axis.text(
                center, -.075, attribute_label,
                transform=axis.get_xaxis_transform(), ha="center", va="top",
                fontsize=22, fontweight=FIG_FONT_WEIGHT, clip_on=False,
            )
            axis.plot(
                [start + .12, end - .12], [-.052, -.052],
                transform=axis.get_xaxis_transform(), color="#777777",
                linewidth=.9, clip_on=False,
            )
        for temperature_label, start, end in temperature_spans:
            center = (start + end) / 2
            axis.text(
                center, -.155, temperature_label,
                transform=axis.get_xaxis_transform(), ha="center", va="top",
                fontsize=24, fontweight=FIG_FONT_WEIGHT, clip_on=False,
            )
            axis.plot(
                [start + .12, end - .12], [-.130, -.130],
                transform=axis.get_xaxis_transform(), color="#333333",
                linewidth=1.2, clip_on=False,
            )

        for row_index in range(1, len(combined_row_order)):
            previous = combined_row_order[row_index - 1]
            current = combined_row_order[row_index]
            if previous in spacer_rows or current in spacer_rows:
                continue
            if current.split(" | ")[0] != previous.split(" | ")[0]:
                axis.hlines(row_index, *axis.get_xlim(), colors="black", linewidth=1.6)
        for boundary in attribute_boundaries:
            axis.vlines(boundary, *axis.get_ylim(), colors="#666666", linewidth=.9)
        for boundary in temperature_boundaries:
            axis.vlines(boundary, *axis.get_ylim(), colors="black", linewidth=1.8)

        spacer_positions = [
            index for index, label in enumerate(combined_row_order) if label in spacer_rows
        ]
        if spacer_positions:
            axis.add_patch(plt.Rectangle(
                (0, min(spacer_positions)), len(column_order), len(spacer_positions),
                facecolor="white", edgecolor="none", zorder=20, clip_on=False,
            ))

    colorbar = heat_axes[1].collections[0].colorbar
    colorbar.ax.tick_params(labelsize=20)
    colorbar.set_label("MSD (LLM − survey)", fontsize=19, weight=FIG_FONT_WEIGHT)
    for axis in heat_axes:
        axis.set_yticks(np.arange(len(display_row_labels)) + .5)
    heat_axes[0].set_yticklabels(display_row_labels, rotation=0)
    heat_axes[0].tick_params(axis="y", labelsize=24, pad=6, colors="#333333")
    heat_axes[1].tick_params(axis="y", labelleft=False, length=0)

    # Panel b: original model-column CDF grid, now using canonical model colors.
    cdf_rows = [
        ("dem", "dem", "Democratic exposure", "Dem egos"),
        ("dem", "rep", "Democratic exposure", "Rep egos"),
        ("rep", "dem", "Republican exposure", "Dem egos"),
        ("rep", "rep", "Republican exposure", "Rep egos"),
    ]
    cdf_grid = outer[1].subgridspec(len(cdf_rows), len(model_order), hspace=.15, wspace=.07)
    cdf_axes = np.empty((len(cdf_rows), len(model_order)), dtype=object)

    for row_index, (party, ego_party, exposure_title, ego_title) in enumerate(cdf_rows):
        for column_index, model_label in enumerate(model_order):
            shared = cdf_axes[0, 0] if (row_index > 0 or column_index > 0) else None
            axis = fig.add_subplot(cdf_grid[row_index, column_index], sharex=shared, sharey=shared)
            cdf_axes[row_index, column_index] = axis
            model_runs = view_exposure[
                view_exposure["model_label"].eq(model_label)
                & view_exposure["party"].eq(party)
                & view_exposure["ego_party"].eq(ego_party)
            ]
            run_cdfs = []
            for _, run in model_runs.groupby("run_id", observed=True):
                cdf = cumulative_distribution(run["llm_cat"])
                run_cdfs.append(cdf)
                axis.plot(
                    cdf.index, cdf.values, color=EXPOSURE_COLORS[party],
                    alpha=.065, linewidth=.9,
                )
            average_cdf = pd.concat(run_cdfs, axis=1).mean(axis=1)
            axis.plot(
                average_cdf.index, average_cdf.values, marker="o",
                linewidth=3.2, markersize=6.7, color=EXPOSURE_COLORS[party], zorder=5,
            )

            survey_values = survey.loc[
                survey["ego_party"].eq(ego_party), f"survey_cat_{party}"
            ]
            survey_cdf = cumulative_distribution(survey_values)
            axis.plot(
                survey_cdf.index, survey_cdf.values, marker="o",
                linewidth=3.2, markersize=6.7, color=SURVEY_COLOR, zorder=6,
            )
            axis.set_xlabel("")
            axis.set_ylabel("CDF", fontsize=24, fontweight=FIG_FONT_WEIGHT, labelpad=7)
            axis.set_xticks(BIN_CATS)
            axis.set_ylim(0, 1.02)
            axis.tick_params(axis="both", labelsize=24, pad=2)
            if column_index != 0:
                axis.tick_params(axis="y", labelleft=False)
            if row_index != len(cdf_rows) - 1:
                axis.tick_params(axis="x", labelbottom=False)
            axis.grid(True, alpha=.18)
            axis.set_box_aspect(.76)

    fig.subplots_adjust(left=.175, right=.965, top=.945, bottom=.090)
    fig.canvas.draw()
    heat_left = min(axis.get_position().x0 for axis in heat_axes)
    heat_top = max(axis.get_position().y1 for axis in heat_axes)
    cdf_left = min(axis.get_position().x0 for axis in cdf_axes.flat)
    cdf_top = max(axis.get_position().y1 for axis in cdf_axes.flat)
    cdf_right = max(axis.get_position().x1 for axis in cdf_axes.flat)
    cdf_bottom = min(axis.get_position().y0 for axis in cdf_axes.flat)

    panel_letter_x = heat_left - .055
    fig.text(panel_letter_x, heat_top + .020, "a", ha="left", va="bottom", fontsize=48)
    fig.text(panel_letter_x, cdf_top + .026, "b", ha="left", va="bottom", fontsize=48)
    fig.text(
        .965, .982, VIEW_LABELS[network_view], ha="right", va="top",
        fontsize=22, color="#333333",
    )
    fig.text(
        (cdf_left + cdf_right) / 2, cdf_bottom - .027, "Exposure category",
        ha="center", va="top", fontsize=26, color="black",
    )

    for column_index, model_label in enumerate(model_order):
        center = (
            cdf_axes[0, column_index].get_position().x0
            + cdf_axes[0, column_index].get_position().x1
        ) / 2
        fig.text(
            center, cdf_top + .010, model_label, ha="center", va="bottom",
            fontsize=24, color="#333333", fontweight=FIG_FONT_WEIGHT,
        )
    for boundary in size_model_boundaries:
        left = max(cdf_axes[row, boundary - 1].get_position().x1 for row in range(len(cdf_rows)))
        right = min(cdf_axes[row, boundary].get_position().x0 for row in range(len(cdf_rows)))
        separator = (left + right) / 2
        fig.add_artist(plt.Line2D(
            [separator, separator], [cdf_bottom, cdf_top],
            transform=fig.transFigure, color="#555555", linewidth=1.5,
        ))

    ego_label_x = cdf_left - .072
    bracket_x = cdf_left - .102
    exposure_label_x = cdf_left - .136
    for row_index, (_, _, _, ego_title) in enumerate(cdf_rows):
        top = max(cdf_axes[row_index, col].get_position().y1 for col in range(len(model_order)))
        bottom = min(cdf_axes[row_index, col].get_position().y0 for col in range(len(model_order)))
        fig.text(
            ego_label_x, (top + bottom) / 2, ego_title,
            ha="center", va="center", rotation=90, fontsize=24, color="#333333",
        )
    for rows, exposure_title in [((0, 1), "Democratic exposure"), ((2, 3), "Republican exposure")]:
        top = max(cdf_axes[rows[0], col].get_position().y1 for col in range(len(model_order)))
        bottom = min(cdf_axes[rows[1], col].get_position().y0 for col in range(len(model_order)))
        fig.add_artist(plt.Line2D(
            [bracket_x, bracket_x], [bottom + .002, top - .002],
            transform=fig.transFigure, color="#555555", linewidth=1.25,
        ))
        for y in (top - .002, bottom + .002):
            fig.add_artist(plt.Line2D(
                [bracket_x, bracket_x + .006], [y, y],
                transform=fig.transFigure, color="#555555", linewidth=1.25,
            ))
        fig.text(
            exposure_label_x, (top + bottom) / 2, exposure_title,
            ha="center", va="center", rotation=90, fontsize=24, color="black",
        )

    legend_handles = [
        plt.Line2D(
            [0], [0], color=EXPOSURE_COLORS["dem"], marker="o",
            linewidth=2.8, markersize=8, label="LLM average: Democratic exposure",
        ),
        plt.Line2D(
            [0], [0], color=EXPOSURE_COLORS["rep"], marker="o",
            linewidth=2.8, markersize=8, label="LLM average: Republican exposure",
        ),
        plt.Line2D(
            [0], [0], color=SURVEY_COLOR, marker="o",
            linewidth=2.8, markersize=8, label="ANES",
        ),
    ]
    fig.legend(
        handles=legend_handles, loc="lower center", ncol=3,
        bbox_to_anchor=((cdf_left + cdf_right) / 2, .024),
        bbox_transform=fig.transFigure, borderaxespad=0,
        fontsize=19, handlelength=1.55, handletextpad=.35,
        columnspacing=.9, frameon=False,
    )

    stem = FIGURE_DIR / f"Fig_3_sample_100_v3_{network_view}"
    fig.savefig(stem.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(stem.with_suffix(".png"), dpi=300, bbox_inches="tight")
    plt.show()
    return fig


directed_figure = make_exposure_figure("directed_out")
mutual_figure = make_exposure_figure("mutual")


In [ ]:
# Export the directed-out figure without its top-right network-view label.
view_label = next(
    (text for text in directed_figure.texts if text.get_text() == VIEW_LABELS["directed_out"]),
    None,
)
if view_label is None:
    raise RuntimeError("Directed-out network-view label not found.")

view_label.set_visible(False)
clean_pdf = FIGURE_DIR / "Fig_3_sample_100_v3_directed_out_no_view_label.pdf"
directed_figure.savefig(clean_pdf, bbox_inches="tight")
view_label.set_visible(True)
print(f"Saved {clean_pdf.relative_to(BASE_DIR)}")


In [ ]:
# Compact paper version: retain panel a and summarize panel b in four CDF plots.
PAPER_MODEL_COLORS = {
    "Llama-3.1 8B": "#793B56",
    "Mistral 7B": "#B45791",
    "Qwen2.5 7B": "#EF895F",
    "Llama-3.1 70B AWQ": "#86CEB0",
    "Qwen2.5 72B AWQ": "#078BA8",
}


def make_compact_exposure_figure(network_view="directed_out"):
    view_metrics = run_metrics[
        run_metrics["network_view"].eq(network_view)
        & run_metrics["model_label"].isin(MODEL_ORDER)
    ].copy()
    view_exposure = exposure_long_all[
        exposure_long_all["network_view"].eq(network_view)
        & exposure_long_all["model_label"].isin(MODEL_ORDER)
    ].copy()
    model_order = [model for model in MODEL_ORDER if model in set(view_metrics["model_label"])]

    view_metrics["method"] = pd.Categorical(
        view_metrics["method"].astype(str), METHOD_ORDER, ordered=True
    )
    view_metrics["row_label"] = (
        view_metrics["model_label"] + " | " + view_metrics["method"].astype(str)
    )
    view_metrics["column_label"] = (
        "T=" + view_metrics["temperature"].map(lambda x: f"{x:g}")
        + " | " + view_metrics["attr_label"].astype(str)
        + " | " + view_metrics["ego_party"].map(EGO_LABELS).astype(str)
    )
    row_frame = view_metrics[["model_label", "method", "row_label"]].drop_duplicates()
    row_frame["model_label"] = pd.Categorical(
        row_frame["model_label"], model_order, ordered=True
    )
    row_order = row_frame.sort_values(["model_label", "method"])["row_label"].tolist()
    column_order = (
        view_metrics[["temperature", "attr_label", "ego_party", "column_label"]]
        .drop_duplicates()
        .assign(
            attr_rank=lambda frame: frame["attr_label"].map({"vote-first": 0, "vote-last": 1}),
            ego_rank=lambda frame: frame["ego_party"].map({"dem": 0, "other": 1, "rep": 2}),
        )
        .sort_values(["temperature", "attr_rank", "ego_rank"])["column_label"]
        .tolist()
    )

    model_groups = []
    for group_label, group_models in MODEL_SIZE_GROUPS:
        present = [model for model in group_models if model in model_order]
        if present:
            model_groups.append((group_label, present))
    combined_rows, spacer_rows = [], []
    for group_index, (_, group_models) in enumerate(model_groups):
        if group_index:
            spacer = f"__compact_spacer_{group_index}__"
            spacer_rows.append(spacer)
            combined_rows.append(spacer)
        combined_rows.extend(
            row for row in row_order if row.split(" | ")[0] in set(group_models)
        )
    display_rows = ["" if row in spacer_rows else row.split(" | ")[1] for row in combined_rows]
    model_spans = []
    for model_label in model_order:
        positions = [
            index for index, row in enumerate(combined_rows)
            if row not in spacer_rows and row.split(" | ")[0] == model_label
        ]
        model_spans.append((model_label, min(positions), max(positions) + 1))

    column_parts = [label.split(" | ") for label in column_order]
    ego_ticks = [parts[2] for parts in column_parts]
    attribute_spans = contiguous_spans([parts[1] for parts in column_parts])
    temperature_spans = contiguous_spans([parts[0] for parts in column_parts])

    fig = plt.figure(figsize=(22.5, 17.5), constrained_layout=False)
    outer = fig.add_gridspec(2, 1, height_ratios=[1.55, .55], hspace=.38)
    heat_grid = outer[0].subgridspec(1, 3, width_ratios=[1, 1, .018], wspace=.045)
    heat_axes = []
    colorbar_axis = fig.add_subplot(heat_grid[0, 2])

    for panel_index, (party, title) in enumerate([
        ("dem", "Democratic exposure"), ("rep", "Republican exposure")
    ]):
        matrix = (
            view_metrics[view_metrics["party"].eq(party)]
            .pivot_table(
                index="row_label", columns="column_label", values="signed_error",
                aggfunc="mean", observed=True,
            )
            .reindex(index=combined_rows, columns=column_order)
        )
        axis = fig.add_subplot(
            heat_grid[0, panel_index], sharey=heat_axes[0] if heat_axes else None
        )
        heat_axes.append(axis)
        sns.heatmap(
            matrix, mask=matrix.isna(), annot=True, fmt=".2f",
            cmap="PiYG", center=0, vmin=-1.8, vmax=1.8,
            linewidths=.34, linecolor="white", cbar=(panel_index == 1),
            cbar_ax=colorbar_axis if panel_index == 1 else None,
            annot_kws={"size": 16, "weight": FIG_FONT_WEIGHT, "fontfamily": "Barlow"},
            ax=axis,
        )
        axis.set_facecolor("white")
        axis.set_title(title, fontsize=31, pad=18, fontweight=FIG_FONT_WEIGHT)
        axis.set_xlabel("")
        axis.set_ylabel("")
        axis.set_xticks(np.arange(len(column_order)) + .5, ego_ticks, rotation=0)
        axis.tick_params(axis="x", labelsize=20, pad=5, colors="#333333")
        axis.set_yticks(np.arange(len(display_rows)) + .5)
        if panel_index == 0:
            axis.set_yticklabels(display_rows, rotation=0)
            axis.tick_params(axis="y", labelsize=24, pad=6, colors="#333333")
            for model_label, start, end in model_spans:
                axis.text(
                    -.150, (start + end) / 2, model_label,
                    transform=axis.get_yaxis_transform(), ha="right", va="center",
                    fontsize=24, fontweight="semibold", color="#333333", clip_on=False,
                )
                axis.plot(
                    [-.088, -.102, -.102, -.088], [start + .15, start + .15, end - .15, end - .15],
                    transform=axis.get_yaxis_transform(), color="#555555",
                    linewidth=1.2, clip_on=False,
                )
        else:
            axis.tick_params(axis="y", labelleft=False, length=0)

        for attribute_label, start, end in attribute_spans:
            axis.text(
                (start + end) / 2, -.075, attribute_label,
                transform=axis.get_xaxis_transform(), ha="center", va="top",
                fontsize=22, clip_on=False,
            )
            axis.plot(
                [start + .12, end - .12], [-.052, -.052],
                transform=axis.get_xaxis_transform(), color="#777777",
                linewidth=.9, clip_on=False,
            )
        for temperature_label, start, end in temperature_spans:
            axis.text(
                (start + end) / 2, -.155, temperature_label,
                transform=axis.get_xaxis_transform(), ha="center", va="top",
                fontsize=24, clip_on=False,
            )
            axis.plot(
                [start + .12, end - .12], [-.130, -.130],
                transform=axis.get_xaxis_transform(), color="#333333",
                linewidth=1.2, clip_on=False,
            )
        for row_index in range(1, len(combined_rows)):
            previous, current = combined_rows[row_index - 1], combined_rows[row_index]
            if previous not in spacer_rows and current not in spacer_rows:
                if current.split(" | ")[0] != previous.split(" | ")[0]:
                    axis.hlines(row_index, *axis.get_xlim(), colors="black", linewidth=1.6)
        for boundary in [end for _, _, end in attribute_spans[:-1]]:
            axis.vlines(boundary, *axis.get_ylim(), colors="#666666", linewidth=.9)
        for boundary in [end for _, _, end in temperature_spans[:-1]]:
            axis.vlines(boundary, *axis.get_ylim(), colors="black", linewidth=1.8)
        spacer_positions = [index for index, row in enumerate(combined_rows) if row in spacer_rows]
        if spacer_positions:
            axis.add_patch(plt.Rectangle(
                (0, min(spacer_positions)), len(column_order), len(spacer_positions),
                facecolor="white", edgecolor="none", zorder=20, clip_on=False,
            ))

    colorbar = heat_axes[1].collections[0].colorbar
    colorbar.ax.tick_params(labelsize=20)
    colorbar.set_label("MSD (LLM − survey)", fontsize=20)
    for axis in heat_axes:
        axis.set_yticks(np.arange(len(display_rows)) + .5)
    heat_axes[0].set_yticklabels(display_rows, rotation=0)
    heat_axes[0].tick_params(axis="y", labelsize=24, pad=6, colors="#333333")
    heat_axes[1].tick_params(axis="y", labelleft=False, length=0)

    cdf_specs = [
        ("dem", "dem", "Dem ego → Dem exposure"),
        ("dem", "rep", "Rep ego → Dem exposure"),
        ("rep", "dem", "Dem ego → Rep exposure"),
        ("rep", "rep", "Rep ego → Rep exposure"),
    ]
    cdf_grid = outer[1].subgridspec(1, 4, wspace=.22)
    cdf_axes = []
    for plot_index, (party, ego_party, title) in enumerate(cdf_specs):
        axis = fig.add_subplot(
            cdf_grid[0, plot_index],
            sharex=cdf_axes[0] if cdf_axes else None,
            sharey=cdf_axes[0] if cdf_axes else None,
        )
        cdf_axes.append(axis)
        survey_cdf = cumulative_distribution(
            survey.loc[survey["ego_party"].eq(ego_party), f"survey_cat_{party}"]
        )
        axis.plot(
            survey_cdf.index, survey_cdf.values, color=SURVEY_COLOR,
            marker="o", linewidth=3.3, markersize=7.2, zorder=7,
        )
        for model_label in model_order:
            model_runs = view_exposure[
                view_exposure["model_label"].eq(model_label)
                & view_exposure["party"].eq(party)
                & view_exposure["ego_party"].eq(ego_party)
            ]
            run_cdfs = [
                cumulative_distribution(run["llm_cat"])
                for _, run in model_runs.groupby("run_id", observed=True)
            ]
            mean_cdf = pd.concat(run_cdfs, axis=1).mean(axis=1)
            axis.plot(
                mean_cdf.index, mean_cdf.values, color=PAPER_MODEL_COLORS[model_label],
                marker="o", linewidth=2.7, markersize=6.2, alpha=.98, zorder=5,
            )
        axis.set_title(title, fontsize=24, pad=10)
        axis.set_xticks(BIN_CATS)
        axis.set_ylim(0, 1.02)
        axis.set_ylabel("CDF", fontsize=22)
        axis.set_xlabel("Exposure category", fontsize=22)
        axis.tick_params(axis="both", labelsize=20)
        axis.grid(True, alpha=.18)
        axis.spines[["top", "right"]].set_visible(False)
        axis.set_box_aspect(1)

    fig.subplots_adjust(left=.175, right=.965, top=.965, bottom=.105)
    fig.canvas.draw()
    heat_left = min(axis.get_position().x0 for axis in heat_axes)
    heat_top = max(axis.get_position().y1 for axis in heat_axes)
    cdf_left = min(axis.get_position().x0 for axis in cdf_axes)
    cdf_top = max(axis.get_position().y1 for axis in cdf_axes)
    panel_letter_x = heat_left - .055
    fig.text(panel_letter_x, heat_top + .018, "a", fontsize=48, ha="left", va="bottom")
    fig.text(panel_letter_x, cdf_top + .018, "b", fontsize=48, ha="left", va="bottom")

    legend_handles = [
        plt.Line2D([0], [0], color=SURVEY_COLOR, marker="o", linewidth=3, label="ANES")
    ] + [
        plt.Line2D(
            [0], [0], color=PAPER_MODEL_COLORS[model], marker="o",
            linewidth=2.6, label=model,
        )
        for model in model_order
    ]
    fig.legend(
        handles=legend_handles, loc="lower center", ncol=6,
        bbox_to_anchor=(.57, .025), frameon=False, fontsize=19,
        handlelength=1.6, handletextpad=.4, columnspacing=1.1,
    )

    image_dir = BASE_DIR / "code" / "images"
    image_dir.mkdir(parents=True, exist_ok=True)
    output_name = {
        "directed_out": "Fig_3_sample_100_v3_directed_out_compact.pdf",
        "mutual": "Fig_S3_partisan_exposure_reciprocal.pdf",
    }[network_view]
    output_path = image_dir / output_name
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    print(f"Saved {output_path.relative_to(BASE_DIR)}")
    return fig


compact_directed_figure = make_compact_exposure_figure("directed_out")
compact_reciprocal_figure = make_compact_exposure_figure("mutual")


## Outputs

- Persona-level exposure data for notebooks 04–06.
- Run-level metrics and ten-run summaries.
- Run-level coverage diagnostics.
- Structural-plausibility and paired network-view robustness tables.
- Tables S7 and S8 and the primary and reciprocal exposure figures.


In [ ]:
for path in [
    EXPOSURE_LONG_CSV, VIEW_METRICS_CSV, SUMMARY_CSV, VIEW_COVERAGE_CSV,
    PLAUSIBILITY_EXPOSURE_CSV, PAIRED_VIEW_CSV,
    BASE_DIR / "code" / "tables" / "table_S7_partisan_exposure_nomination.tex",
    BASE_DIR / "code" / "tables" / "table_S8_partisan_exposure_reciprocal.tex",
    FIGURE_DIR / "Fig_3_sample_100_v3_directed_out.pdf",
    FIGURE_DIR / "Fig_3_sample_100_v3_mutual.pdf",
    BASE_DIR / "code" / "images" / "Fig_S3_partisan_exposure_reciprocal.pdf",
]:
    print(f"{path.relative_to(BASE_DIR)}: {path.stat().st_size / 1024:.1f} KiB")
